# Overview

In [1]:
%matplotlib inline

import pandas as pd
import importlib
import re
import warnings
import calendar
import copy
import oceanval
import pickle

import glob
import geopandas as gpd
from oceanval.quiet_nc import nc
import os
import pickle
import numpy as np
import xarray as xr
from IPython.display import Markdown as md_markdown
from tqdm import tqdm
from IPython.core.interactiveshell import InteractiveShell
from oceanval.tidiers import tidy_info
from oceanval.map_land import land_below
InteractiveShell.ast_node_interactivity = "all"

try:
    lon_lim = None
    lat_lim = None
except:
    lon_lim = None
    lat_lim = None
if lon_lim is None:
    lon_lim = [-180, 180]
if lat_lim is None:
    lat_lim = [-90, 90]

try:
    concise = False
except:
    concise = False

# the depth ranges point matchups through the water column are summarised in
from oceanval.depths import DEFAULT_DEPTH_BINS, bin_depth, depth_labels, depth_ranges_text
try:
    depth_bins = the_depth_bins
except:
    depth_bins = DEFAULT_DEPTH_BINS
depth_levels = depth_labels(depth_bins)

# in one line
from oceanval.tidiers import fix_unit, df_display, md, md_basic

warnings.filterwarnings('ignore')

from oceanval.utils import restrict_r_to_conda
restrict_r_to_conda()

%load_ext rpy2.ipython

test_status = False


i_figure = 1
i_table = 1
stamp = nc.session_info["stamp"]
out = ".trackers/" + stamp
fast_plot = False
if not os.path.exists(".trackers"):
    os.makedirs(".trackers")
# save out as empty file
with open(out, 'w') as f:
    f.write("")

def bin_value(x, bin_res):
    return np.floor((x + bin_res / 2) / bin_res + 0.5) * bin_res - bin_res / 2

try:
    vv_name = variable
except:
    vv_name = "summary"
    variable = "summary"
Variable = variable.title()

In [2]:
import datetime
from oceanval import sim_compare
from oceanval import sim_plots as sp
from oceanval import sim_report as sr
from oceanval import analysis_plots as ap

# what compare_simulations() saved, beside the report
results_dir = "../../results"
info = sim_compare.load(results_dir)
sims = [sim["name"] for sim in info["simulations"]]
by_name = {sim["name"]: sim for sim in info["simulations"]}
years_text = sr.years_text

In [3]:
created = datetime.datetime.fromisoformat(info["created"])
count = len(info["variables"])
md(
    f"This report compares {len(sims)} simulations: {sr.colour_names(info)}, with no observations. "
    f"It covers {count} variable{'s' if count != 1 else ''}, and was made with **oceanval** "
    f"{info['oceanval']} on {created.day} {created.strftime('%B %Y')}. "
    "Each variable has a page of its own, listed on the left, in which the simulations are shown side by side, "
    "each in the colour it has in the tables below."
)

This report compares 2 simulations: **fv** and **nm**, with no observations. It covers 1 variable, and was made with **oceanval** 0.6.1 on 3 November 2026. Each variable has a page of its own, listed on the left, in which the simulations are shown side by side, each in the colour it has in the tables below.

In [4]:
rows = []
for sim in info["simulations"]:
    rows.append(
        {
            "Simulation": sr.swatch(info, sim["name"]),
            "Directory": sim["sim_dir"],
            "Directories down": sim["n_dirs_down"],
            "Thickness": sim["thickness"] or "None",
        }
    )
pd.DataFrame(rows).style.hide(axis="index")

Simulation,Directory,Directories down,Thickness
fv,/data/thaumus2/scratch/maz/Models/FVCOM/run/aqua_v16_trawl/output/aqua_v16_trawl/tide_wind_nest_ersem/2012_2014_val,0,z_level
nm,/data/thaumus2/scratch/hpo/COMFORT/baseline_archerfull/2013,1,None


In [5]:
md(f"**Table {i_table}:** The simulations, the directories their output is in, and how many directories below those the files are.")
i_table += 1

**Table 1:** The simulations, the directories their output is in, and how many directories below those the files are.

In [6]:
rows = []
for variable in info["variables"]:
    row = {"Variable": variable["name"], "Unit": ap.unit_html(variable["units"])}
    for sim in sims:
        own = variable["sims"][sim]
        row[sr.swatch(info, sim)] = f"<code>{own['model_variable'].replace('+', ' + ')}</code>"
    rows.append(row)
pd.DataFrame(rows).style.hide(axis="index")

Variable,Unit,fv,nm
Temperature,degrees_C,temp,votemper


In [7]:
md(f"**Table {i_table}:** The variables compared, and the model variable holding each in each simulation.")
i_table += 1

**Table 2:** The variables compared, and the model variable holding each in each simulation.

In [8]:
rows = []
for variable in info["variables"]:
    for sim in sims:
        own = variable["sims"][sim]
        rows.append(
            {
                "Variable": variable["name"],
                "Simulation": sr.swatch(info, sim),
                "Files": f"{own['pattern']} ({own['files']})",
                "Output": own["time_res_text"].capitalize(),
                "Dates": f"{own['first']} to {own['last']}",
            }
        )
pd.DataFrame(rows).style.hide(axis="index")

Variable,Simulation,Files,Output,Dates
Temperature,fv,aqua_v**_trawl_avg_**_**_**.nc (13),Daily,2012-12-02 to 2014-01-01
Temperature,nm,amm7_1d_**_**_grid_T.nc (12),Daily,2013-01-01 to 2013-12-31


In [9]:
md(f"**Table {i_table}:** The output each simulation's variables are read from: the files, named with their dates left out, and how many there are.")
i_table += 1

**Table 3:** The output each simulation's variables are read from: the files, named with their dates left out, and how many there are.

In [10]:
rows = []
for variable in info["variables"]:
    own = variable["sims"][sims[0]]
    clim = own.get("climatology")
    phen = own.get("phenology")
    long_term = own.get("long_term")
    seasonal = own.get("seasonal")
    row = {
        "Variable": variable["name"],
        "Climatology": (
            f"{years_text(clim)}: " + ", ".join(layer["title"].lower() for layer in clim["layers"].values())
            if clim else "None"
        ),
        "Phenology": f"{years_text(phen)}: surface" if phen else "None",
    }
    if any("long_term" in x["kinds"] for x in info["variables"]):
        row["Long-term series"] = (
            f"{years_text(long_term)}: " + ", ".join(layer["title"].lower() for layer in long_term["layers"].values())
            if long_term else "None"
        )
    if any("seasonal" in x["kinds"] for x in info["variables"]):
        row["Seasonal cycles"] = (
            f"{years_text(seasonal)}: " + ", ".join(layer["title"].lower() for layer in seasonal["layers"].values())
            if seasonal else "None"
        )
    if any("map_changes" in x["kinds"] for x in info["variables"]):
        changes = own.get("map_changes")
        row["Map changes"] = (
            ", ".join(f"{period['start']}-{period['end']}" for period in changes["periods"])
            + ": " + ", ".join(layer["title"].lower() for layer in changes["layers"].values())
            if changes else "None"
        )
    if any("transects" in x["kinds"] for x in info["variables"]):
        transects = own.get("transects")
        row["Transects"] = f"{years_text(transects)}: {len(transects['lines'])}" if transects else "None"
    rows.append(row)
pd.DataFrame(rows).style.hide(axis="index")

Variable,Climatology,Phenology
Temperature,2013: surface,None


In [11]:
md(f"**Table {i_table}:** What was worked out for each variable in every simulation, and over which years.")
i_table += 1

**Table 4:** What was worked out for each variable in every simulation, and over which years.

In [12]:
md("## Model grids")
# with a domain mask and no unify, the mask's grid is the one grid, as its sentence says
if info["unify"] is not None or not info.get("domain_mask_text"):
    md(info["unify_text"])
if info.get("domain_mask_text"):
    md(info["domain_mask_text"])
if info["unify"] is None:
    if info["same_grid"] is True:
        md("The simulations are on the same " + ("mesh" if info.get("fvcom") else "grid") + ".")
    elif info["same_grid"] is False:
        md(
            "**The simulations are not all on the same grid**, and they were not put on one: the maps are of each simulation's own grid, "
            "and the means over the domain, the regions and the transects are over the cells each simulation has, "
            "so they are not over exactly the same area."
        )
    else:
        md("Whether the simulations are on the same grid could not be told.")
else:
    rows = []
    for variable in info["variables"]:
        for layer, (shared, cells) in (info["shared_cells"] or {}).get(variable["key"], {}).items():
            rows.append(
                {
                    "Variable": variable["name"],
                    "Layer": layer if layer == "levels" else ap.LAYER_TITLES[layer] if hasattr(ap, "LAYER_TITLES") else layer,
                    "Cells with values in every simulation": f"{shared} of {cells}",
                }
            )
    if rows:
        display(pd.DataFrame(rows).style.hide(axis="index"))
        md(f"**Table {i_table}:** The number of cells of the common grid with a value in every simulation, which are the cells all the simulations are compared over. A cell with no value in one simulation is missing in all of them.")
        i_table += 1

## Model grids

The simulations were put on the grid of nm: each of the others was regridded to it (bilinear interpolation, nctoolkit's regrid). Where any simulation has no value in a cell (judged from its second time step), that cell is missing in all of them, so every simulation is compared over exactly the same cells. The output of fv is on a mesh of triangles, so it was regridded (nearest neighbour) onto the grid of nm, cropped to the extent of the mesh, where only the cells inside the mesh have values. Variables below the surface were interpolated from the model's sigma layers to the depths of the levels of nm, with the top layer's value above the top layer, the bottom layer's between it and the seabed and nothing below the seabed, so the bottom is the deepest of those levels that is above the seabed.

Variable,Layer,Cells with values in every simulation
Temperature,surface,4490 of 13530


**Table 5:** The number of cells of the common grid with a value in every simulation, which are the cells all the simulations are compared over. A cell with no value in one simulation is missing in all of them.

In [13]:
notes = []
for variable in info["variables"]:
    for sim in sims:
        own = variable["sims"][sim]
        for kind in ("climatology", "phenology", "long_term", "seasonal", "transects"):
            analysis = own.get(kind)
            if analysis and analysis["dates"]["issues"]:
                notes.append(f"- **{variable['name']}**, {sim}, {kind.replace('_', '-')} ({years_text(analysis)}): " + " ".join(analysis["dates"]["issues"]))
        changes = own.get("map_changes")
        for period in changes["periods"] if changes else []:
            if period["dates"]["issues"]:
                notes.append(f"- **{variable['name']}**, {sim}, map changes, {period['title'].lower()} period ({years_text(period)}): " + " ".join(period["dates"]["issues"]))
md("## Gaps in the output")
if notes:
    md("The output is missing times in the years some of the variables were analysed for. Averages are taken over the times there are.\n\n" + "\n".join(notes))
else:
    md("No times are missing from the output of any simulation in the years each variable was analysed for.")

## Gaps in the output

No times are missing from the output of any simulation in the years each variable was analysed for.

In [14]:
md("## Methods")
paragraphs = sr.methods(info)
at = 1
if info["unify"] is not None or not info.get("domain_mask_text"):
    paragraphs.insert(at, "**Model grids.** " + info["unify_text"])
    at += 1
if info.get("domain_mask_text"):
    paragraphs.insert(at, "**Domain mask.** " + info["domain_mask_text"])
thicknesses = {sim["name"]: sim["thickness"] for sim in info["simulations"] if sim["thickness"]}
if thicknesses:
    paragraphs.append(
        "The cell thickness of the water column is "
        + ("on fixed depth levels (z-levels), whose thicknesses are taken from the levels themselves" if set(thicknesses.values()) == {"z_level"} else
           "; ".join(f"`{x}` in {sim}" if x != "z_level" else f"the levels themselves in {sim}" for sim, x in thicknesses.items()))
        + ", at its first time if it changes over time."
    )
missing = {sim["name"]: sim["as_missing"] for sim in info["simulations"] if sim["as_missing"] is not None}
if missing:
    def what(value):
        if isinstance(value, list):
            return f"{value[0]:g}" if value[0] == value[1] else f"{value[0]:g} to {value[1]:g}"
        return f"{value:g}"
    paragraphs.append(
        "Values in the model's output were treated as missing, as were those of the cell thickness: "
        + "; ".join(f"{what(value)} in {sim}" for sim, value in missing.items()) + "."
    )
shifted = {sim["name"]: sim["time_shift"] for sim in info["simulations"] if sim["time_shift"]}
if shifted:
    paragraphs.append(
        "The model's times were moved before they were used, as they mark the end of the period each mean covers: "
        + "; ".join(f"{sim} by {days} day{'s' if abs(days) != 1 else ''}" for sim, days in shifted.items()) + "."
    )
md("\n\n".join(paragraphs))

## Methods

**The simulations.** Every simulation is analysed in the same way, for the same years, from its own output, and shown beside the others, in a colour for each.

**Model grids.** The simulations were put on the grid of nm: each of the others was regridded to it (bilinear interpolation, nctoolkit's regrid). Where any simulation has no value in a cell (judged from its second time step), that cell is missing in all of them, so every simulation is compared over exactly the same cells. The output of fv is on a mesh of triangles, so it was regridded (nearest neighbour) onto the grid of nm, cropped to the extent of the mesh, where only the cells inside the mesh have values. Variables below the surface were interpolated from the model's sigma layers to the depths of the levels of nm, with the top layer's value above the top layer, the bottom layer's between it and the seabed and nothing below the seabed, so the bottom is the deepest of those levels that is above the seabed.

**Climatology.** For each variable, the mean of each month of each year is taken first, so that a month of daily output counts as much as a month of monthly output. The climatology is the mean of each month of the year over the years chosen, and the annual mean is the mean of those monthly means. At the surface, the first value from the top of the water column that is not missing is taken, as in OceanVal's matchups. At the bottom, the deepest value that is not missing is taken (nctoolkit's `bottom` method with `choice="value"`). The water column's vertical mean weights each layer by its thickness; its depth integral multiplies each layer by its thickness and sums them, so a concentration per cubic metre becomes an amount per square metre. The maps of the simulations share one colour scale, from the 2<sup>nd</sup> to the 98<sup>th</sup> percentile of all of them.

**Units.** Every variable is compared in the units fv gives it in. A simulation giving it in other units had each value multiplied by a number and had a number added to it as it was read from the output, before anything was worked out from it (values were taken as missing in the simulation's own units, before that): Temperature in nm, from degC into degrees_C (the same units, written otherwise).

The cell thickness of the water column is on fixed depth levels (z-levels), whose thicknesses are taken from the levels themselves, at its first time if it changes over time.

Values in the model's output were treated as missing, as were those of the cell thickness: 0 in fv; 0 in nm.

In [15]:
# delete all objects starting with ds
for name in dir():
    if name.startswith("ds"):
        del globals()[name]
for name in dir():
    if name.endswith("mask"):
        del globals()[name]
nc.cleanup()
for ff in nc.session.get_safe():
    nc.session.remove_safe(ff)
nc.cleanup()